# B-Glue Month-to-Quantity Comparison Analysis (B膠資料比對分析)

This notebook uses Python's **standard library only** (no external packages like `pandas` or `openpyxl` required) to compare the B-Glue data between standard reference sheets and our generated sheets.


In [6]:
import zipfile
import xml.etree.ElementTree as ET
import os

def get_shared_strings(z):
    strings = []
    try:
        ss_data = z.read('xl/sharedStrings.xml')
        root = ET.fromstring(ss_data)
        for elem in root.iter():
            if elem.tag.split('}')[-1] == 'si':
                si_text = ''
                for child in elem.iter():
                    if child.tag.split('}')[-1] == 't':
                        si_text += child.text or ''
                strings.append(si_text)
    except KeyError:
        pass
    return strings

def get_sheet_target(z, sheet_name):
    wb_data = z.read('xl/workbook.xml')
    root = ET.fromstring(wb_data)
    sheet_id_rel = None
    for elem in root.iter():
        if elem.tag.split('}')[-1] == 'sheet':
            if elem.attrib.get('name') == sheet_name:
                for k, v in elem.attrib.items():
                    if k.endswith('id'):
                        sheet_id_rel = v
                        break
                break
    if not sheet_id_rel:
        names = [elem.attrib.get('name') for elem in root.iter() if elem.tag.split('}')[-1] == 'sheet']
        raise ValueError(f"Sheet '{sheet_name}' not found. Available sheets: {names}")
    rels_data = z.read('xl/_rels/workbook.xml.rels')
    rels_root = ET.fromstring(rels_data)
    for elem in rels_root.iter():
        if elem.tag.split('}')[-1] == 'Relationship':
            if elem.attrib.get('Id') == sheet_id_rel:
                return elem.attrib.get('Target')
    raise ValueError(f"Relation target for ID {sheet_id_rel} not found")

def read_xlsx_sheet(filename, sheet_name):
    if not os.path.exists(filename):
        return None
    with zipfile.ZipFile(filename, 'r') as z:
        shared_strings = get_shared_strings(z)
        target = get_sheet_target(z, sheet_name)
        path = target if target.startswith('xl/') else 'xl/' + target
        sheet_data = z.read(path)
        root = ET.fromstring(sheet_data)
        rows = {}
        for row in root.iter():
            if row.tag.split('}')[-1] == 'row':
                r_num = int(row.attrib.get('r', 1))
                row_data = {}
                for cell in row.iter():
                    if cell.tag.split('}')[-1] == 'c':
                        cell_ref = cell.attrib.get('r', '')
                        cell_type = cell.attrib.get('t', '')
                        val = ''
                        for child in cell.iter():
                            if child.tag.split('}')[-1] == 'v':
                                val = child.text or ''
                                break
                        if cell_type == 's' and val:
                            val = shared_strings[int(val)]
                        elif val:
                            try:
                                val = float(val) if '.' in val else int(val)
                            except ValueError:
                                pass
                        row_data[cell_ref] = val
                rows[r_num] = row_data
        if not rows:
            return []
        max_row = max(rows.keys())
        def col_to_idx(col):
            idx = 0
            for c in col:
                idx = idx * 26 + (ord(c) - ord('A') + 1)
            return idx - 1
        max_col_idx = 0
        for r_num, row_data in rows.items():
            for cell_ref in row_data:
                col_name = ''.join(filter(str.isalpha, cell_ref))
                c_idx = col_to_idx(col_name)
                if c_idx > max_col_idx: max_col_idx = c_idx
        grid = [['' for _ in range(max_col_idx + 1)] for _ in range(max_row)]
        for r_num, row_data in rows.items():
            for cell_ref, val in row_data.items():
                col_name = ''.join(filter(str.isalpha, cell_ref))
                c_idx = col_to_idx(col_name)
                grid[r_num - 1][c_idx] = val
        return grid

std_file = "DataExtract/2025/進料檢驗-2025.xlsx"
gen_file = "DataExtract/2025品檢報表統計.xlsx"
raw_file = "RawData/2025/進料檢驗-2025/物料/B膠.xlsx"

# 1. Read standard sheet
print("=== 1. Standard Consolidated '進料檢驗-2025.xlsx' (物料-B膠) ===")
std_rows = read_xlsx_sheet(std_file, "物料-B膠")
if std_rows:
    for r in std_rows[:15]:
        print([str(x)[:15] for x in r])
else:
    print("Not found or empty")

# 2. Read generated sheet
print("\n=== 2. Generated '2025品檢報表統計.xlsx' (原物料品檢(QC10002-R02)) ===")
gen_rows = read_xlsx_sheet(gen_file, "原物料品檢(QC10002-R02)")
if gen_rows:
    # Print header and monthly values
    print([str(x) for x in gen_rows[1]])  # Headers
    for r in gen_rows[2:14]:
        # Show month and B膠 column
        headers = gen_rows[1]
        b_idx = headers.index('B膠') if 'B膠' in headers else 1
        print(f"Month {r[0]}: B膠 = {r[b_idx]} (Row subtotal: {r[-1]})")
else:
    print("Not found or empty")

# 3. Read Raw B膠.xlsx sheets
print("\n=== 3. Raw 'B膠.xlsx' Sheets ===")
if os.path.exists(raw_file):
    with zipfile.ZipFile(raw_file, 'r') as z:
        wb_data = z.read('xl/workbook.xml')
        wb_root = ET.fromstring(wb_data)
        sheet_names = [elem.attrib.get('name') for elem in wb_root.iter() if elem.tag.split('}')[-1] == 'sheet']
        print(f"Raw sheets: {sheet_names}")
        for name in sheet_names[:3]:
            print(f"\n--- Sheet: {name} (sample rows) ---")
            rows = read_xlsx_sheet(raw_file, name)
            if rows:
                for r in rows[:6]:
                    print([str(x)[:15] for x in r])
else:
    print("Raw file not found")


=== 1. Standard Consolidated '進料檢驗-2025.xlsx' (物料-B膠) ===
['檔案名稱', '工作表名稱', '表單編碼', '表單名稱', '月份']
['B膠.xlsx', '230807019', 'QC10002-R02', '原物料品檢表', '1']
['B膠.xlsx', '230807020', 'QC10002-R02', '原物料品檢表', '1']
['B膠.xlsx', '230807020 (2)', 'QC10002-R02', '原物料品檢表', '1']
['B膠.xlsx', '230807021', 'QC10002-R02', '原物料品檢表', '1']
['B膠.xlsx', '230807021 (2)', 'QC10002-R02', '原物料品檢表', '3']
['B膠.xlsx', '230807022', 'QC10002-R02', '原物料品檢表', '3']
['B膠.xlsx', '230807022 (2)', 'QC10002-R02', '原物料品檢表', '3']
['B膠.xlsx', '230807022 (3)', 'QC10002-R02', '原物料品檢表', '3']
['B膠.xlsx', '230807023', 'QC10002-R02', '原物料品檢表', '4']
['B膠.xlsx', '230807023 (2)', 'QC10002-R02', '原物料品檢表', '4']
['B膠.xlsx', '230807024', 'QC10002-R02', '原物料品檢表', '4']
['B膠.xlsx', '230807024 (2)', 'QC10002-R02', '原物料品檢表', '5']
['B膠.xlsx', '241101001', 'QC10002-R02', '原物料品檢表', '5']
['B膠.xlsx', '241101002', 'QC10002-R02', '原物料品檢表', '6']

=== 2. Generated '2025品檢報表統計.xlsx' (原物料品檢(QC10002-R02)) ===
['月份', '原料', 'B膠', '收縮膜', '色粉', '空白包裝袋', '空白感壓紙